In [ ]:
%load_ext rpy2.ipython

In [ ]:
import sys, os

import pandas as pd
import numpy as np
from scipy import sparse

import scanpy as sc
import anndata

import matplotlib as mpl
mpl.rc('pdf',fonttype=42)
mpl.rcParams['pdf.use14corefonts'] = True
mpl.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
from natsort import natsorted
from tqdm import tqdm

In [ ]:
barcode2tumor = {
    'A1': '1N',
    'A2': '1L',
    'A3': '1LR',
    'A4': '1RR',
    'B1': '2N',
    'B2': '2L',
    'B3': '2R',
    'B4': '2LR',
    'B5': '2RR'
}

# process MULTI-seq

In [ ]:
%%R
library(deMULTIplex2)

valid.barcodes <- c('A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4', 'B5')
tag.ref <- read.csv('../data/bar.ref.new.csv')
bar.ref <- setNames(as.character(tag.ref$Barcode_Sequence), tag.ref$Well_Position)
valid.bar.ref <- bar.ref[valid.barcodes]

infiles <- list.files(path='../data', pattern="_MULTI_read_table.rds", full.names=TRUE)

for (infile in infiles){
    read_table <- readRDS(infile)
    out.demux <- sub("_read_table\\.rds$", "_demux.rds", infile)
    out.assign_table <- sub("_read_table\\.rds$", "_assign_table.txt", infile)
    
    # use valid oligos
    tag_mtx <- alignTags(read_table, valid.bar.ref)
    
    # demux
    res <- demultiplexTags(tag_mtx,
                           plot.diagnostics=F)

    saveRDS(res, out.demux)
    write.table(demux$assign_table, out.assign_table, sep='\t', quote=F, col.names=NA)
}


# read cellranger filtered matrices

In [ ]:
h5_files = natsorted(glob('../data/*_filtered_feature_bc_matrix.h5'))
h5_files

In [ ]:
for f in h5_files:
    name = os.path.basename(f).split('_filtered_feature_bc_matrix.h5')[0]
    outdir = os.path.join('../results/cellranger')
    os.makedirs(outdir, exist_ok=True)
    print(name)    
    
    local = sc.read_10x_h5(f)
    local.var_names_make_unique()
    
    local.var['mt'] = local.var_names.str.lower().str.startswith('mt-')
    sc.pp.calculate_qc_metrics(local, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    sc.pp.calculate_qc_metrics(local, inplace=True)
    
    valid_cells = (local.obs['pct_counts_mt'] < 10) & (local.obs['n_genes_by_counts'] > 500)
    local.obs['valid_cell'] = valid_cells
    
    print('Cells after filtering: {}/{} ({:.4f})'.format(valid_cells.sum(), local.shape[0], valid_cells.sum()/local.shape[0]))
    pd.Series(local.obs.index[valid_cells].str.split('-').str[0]).to_csv(os.path.join(outdir, 'valid_cells.txt'), 
                                                                         sep='\t', index=False, header=None)
    
    local.raw = local
    local.layers['counts'] = local.X.astype(np.uint32)
    
    # append MULTIseq calls
    multiseq_path = os.path.join('../data/', name, '_MULTI_assign_table.txt')
    assign_table = pd.read_table(multiseq_path, index_col=0)
    assign_table.index = assign_table.index + '-1'
    assign_table['tumor'] = assign_table['barcode_assign'].map(barcode2tumor)
    
    print('writing')
    local.write_h5ad(os.path.join(outdir, name+'_proc.h5ad'))
    

# concatenate

In [ ]:
infiles = natsorted(glob('../data/cellranger/*/*_proc.h5ad'))
infiles

In [ ]:
# load into memory
adatas = {}
for f in infiles:
    name = os.path.basename(f).split('_proc.h5ad')[0]
    print(name)

    local = sc.read_h5ad(f)
    adatas[name] = local.copy()

In [ ]:
adata = anndata.concat(adatas, label='GEM', index_unique='-')
adata

In [ ]:
# recalc metrics
adata.var['mt'] = adata.var_names.str.lower().str.startswith('mt-')
adata.var['syn'] = adata.var_names.str.lower().str.startswith('syn_')

sc.pp.calculate_qc_metrics(adata, layer='counts', qc_vars=['mt', 'syn'], 
                           percent_top=None, log1p=False, inplace=True)

In [ ]:
# normalize counts using size factor
sf = adata.obs['total_counts'] / adata.obs['total_counts'].mean()

adata.obs['size_factor'] = sf
adata.layers['sf_log1p'] = sparse.csr_matrix(adata.layers['counts'] / adata.obs['size_factor'].values.reshape(-1, 1)).log1p()
adata.X = adata.layers['sf_log1p']

In [ ]:
# save result
adata.write_h5ad('../results/adata_raw.h5ad')

# dimensionality reduction

In [ ]:
# append annotations
meta = pd.read_table('../data/cell_metadata.txt', index_col=0)
adata.obs = meta.reindex(adata.obs.index)


In [ ]:
# hard filter pass filter cells
adata = adata[meta.index].copy()
adata.shape

In [ ]:
sc.tl.pca(adata)

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata)

In [ ]:
adata.write_h5ad('../results/adata_proc_singlets.h5ad')

# add match-seq barcodes

In [ ]:
read_table = pd.read_table('../data/MATCH_barcode_table.txt.gz')
read_table

In [ ]:
read_table['label'] = read_table['sgRNA'] + read_table['sgRNA_UMI']
read_table['tumor'] = read_table['cell'].map(adata.obs['tumor'])
read_table['label_tumor'] = read_table['label'] + ':' + read_table['tumor'].astype(str)

In [ ]:
filtered_read_table = read_table[read_table['counts'] > 1].copy()
filtered_read_table.shape

In [ ]:
cell_bcs = adata.obs.index
label_bcs = np.array(natsorted(filtered_read_table['label'].unique()))

In [ ]:
cell_map = {k:i for i,k in enumerate(cell_bcs)}
label_map = {k:i for i,k in enumerate(label_bcs)}

In [ ]:
filtered_read_table['cell_coord'] = filtered_read_table['cell'].map(cell_map)
filtered_read_table['label_coord'] = filtered_read_table['label'].map(label_map)

In [ ]:
cell_coord, label_coord = filtered_read_table[['cell_coord', 'label_coord']].dropna().astype(int).values.T

coo = sparse.coo_matrix((np.ones(len(cell_coord)), (cell_coord, label_coord)), dtype=np.int32)
adj = coo.tocsr()
adj

In [ ]:
adata.obsm['labels'] = adj.copy()
adata.uns['labels'] = label_bcs.copy()

## restrict by tumor

In [ ]:
strict_read_table = filtered_read_table.dropna(subset=['tumor']).copy()

In [ ]:
label_bcs = np.array(natsorted(strict_read_table['label_tumor'].unique()))
label_map = {k:i for i,k in enumerate(label_bcs)}

In [ ]:
strict_read_table['label_coord2'] = strict_read_table['label_tumor'].map(label_map)

In [ ]:
cell_coord, label_coord = strict_read_table[['cell_coord', 'label_coord2']].dropna().astype(int).values.T

coo = sparse.coo_matrix((np.ones(len(cell_coord)), (cell_coord, label_coord)), dtype=np.int32)
adj = coo.tocsr()
adj

In [ ]:
adata.obsm['strict_labels'] = adj.copy()
adata.uns['strict_labels'] = label_bcs_1read.copy()

# make label tables

In [ ]:
celltypes = np.array(natsorted(np.unique(adata.obs['major_celltype'])))
len(celltypes)

In [ ]:
label_groups = pd.Series(adata.uns['labels']).str[:-16]
perturbations = np.array(natsorted(np.unique(label_groups)))
len(perturbations)

In [ ]:
mat = adata.obsm[label_key]
mat

In [ ]:
group_idx, perts = pd.factorize(label_groups)

rows = np.arange(len(group_idx))  # one row per column
cols = group_idx  # each column gets assigned a group index
data = np.ones(len(group_idx))  # sparse matrix values are 1 (one-hot encoding)

design = sparse.csr_matrix((data, (rows, cols)), shape=(len(group_idx), len(perts)))

label_spmat = mat.dot(design)

In [ ]:
label_mat = pd.DataFrame(label_spmat.toarray(), index=adata.obs.index, columns=perturbations).astype(int)
label_mat

In [ ]:
label_mat.to_csv('../results/label_matrix_counts.txt', sep='\t')

## SCT transform

In [ ]:
%%R
library(Seurat)

In [ ]:
%%R -i label_mat

In [ ]:
%%R
nz.labels <- labeling_mat[rowSums(labeling_mat) > 0, ]
dim(nz.labels)

In [ ]:
%%R
labels.so <- CreateSeuratObject(counts = t(nz.labels))
labels.so

In [ ]:
%%R
labels.so <- SCTransform(labels.so, ncells=30000, verbose = TRUE, vst.flavor='v2', return.only.var.genes=FALSE)

In [ ]:
%%R
write.table(t(as.matrix(labels.so@assays$SCT$counts)), '../results/label_matrix_SCT_counts.txt', sep='\t', quote=F, col=NA)


## estimate mean


In [ ]:
def fit_nbinom(y):
    try: 
        intc = np.ones((len(y), 1))
        res = smd.NegativeBinomial(y, intc).fit(max_iter=100, disp=False)
        return np.exp(res.params.loc['const'])
    except:
        return np.nan

In [ ]:
label_mat_sct = pd.read_table('../results/label_matrix_SCT_counts.txt', index_col=0)

In [ ]:
group_labels = meta['major_celltype']

In [ ]:
nbinom_muhat = {}

with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    
    for group in tqdm(label_mat_sct.groupby(group_labels)):
        idx, local = group
        tumor, ct = idx.split(':')

        intc = np.ones((len(local), 1))
        nbinom_muhat[idx] = local.apply(lambda x: fit_nbinom(x), axis=0)

nbinom_est = pd.concat(nbinom_muhat, axis=1)

In [ ]:
nbinom_est.replace(np.inf, np.nan, inplace=True)
nbinom_est.replace(-np.inf, np.nan, inplace=True)

In [ ]:
nbinom_est.T.to_csv('../results/label_matrix_SCT_nbinom_est.txt', sep='\t')